In [2]:
from torch.utils.data import DataLoader
from transformers import pipeline, GenerationConfig
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct, VectorParams, Distance
from dotenv import load_dotenv
import os
from otomoto_ds import OtomotoChunkedDataset
from uuid import uuid4
from utils import *
from models import DINOv2EmbeddingCompressor

In [ ]:
load_dotenv()

COLLECTION_NAME = "used cars dataset"

api_key = os.getenv('QDRANT_API_KEY')
api_url = os.getenv('QDRANT_URL')

client = QdrantClient(
    api_key=api_key,
    url=api_url,
)

if not client.collection_exists(COLLECTION_NAME):
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(
            size=64,
            distance=Distance.COSINE
        )
    )

In [ ]:
llm_model_name = "Qwen/Qwen2.5-3B-Instruct"

llm = pipeline(
    "text-generation",
    model=llm_model_name,
    dtype=torch.float16,
    device_map="auto"
)

In [ ]:
dino_v2_embedder = DINOv2EmbeddingCompressor()

In [6]:
BATCH_SIZE=32
ITEMS=500
CHUNK_SIZE=64


cars_ds = OtomotoChunkedDataset(
    items=ITEMS,
    chunk_size=CHUNK_SIZE
)

dataloader = DataLoader(
    dataset=cars_ds,
    batch_size=BATCH_SIZE,
    collate_fn=collate_fn
)

In [7]:
gen_config = GenerationConfig(
    temperature=0.0,
    do_sample=False,
    max_new_tokens=512
)

def llm_parse(user_content: list[str]) -> list | None:
    output = llm(
        [build_prompt(content) for content in user_content],
        generation_config=gen_config
    )

    return extract_labels(output)

In [ ]:
for images, imgs_number, text, urls in dataloader:
    images = images.to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))

    with torch.no_grad():
        batch_metadata = llm_parse(text)
        embeddings = dino_v2_embedder(images, imgs_number)


    records = []
    for embedding, metadata, url in zip(embeddings, batch_metadata, urls):
        metadata["url"] = url

        for num, vector in enumerate(embedding):
            metadata_copy = metadata.copy()
            metadata_copy["cluster_label"] = num

            records.append(
                PointStruct(
                    id=str(uuid4()),
                    vector=vector,
                    payload=metadata_copy
                )
            )

    if records:
        client.upsert(
            collection_name=COLLECTION_NAME,
            points=records
        )